In [ ]:
# Reading cleaned data from github link
import pandas as pd
customers   = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/olist_customers_dataset.parquet")
geolocation = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/olist_geolocation_dataset.parquet")
order_items = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/olist_order_items_dataset.parquet")
payments    = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/olist_order_payments_dataset.parquet")
reviews     = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/olist_order_reviews_dataset.parquet")
orders      = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/olist_orders_dataset.parquet")
products    = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/olist_products_dataset.parquet")
sellers     = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/olist_sellers_dataset.parquet")
category_translation = pd.read_parquet("https://raw.githubusercontent.com/FarahHossamEldin/E-Commerce-Customer-Value-and-Purchase-Prediction-/main/data/processed/parquet/product_category_name_translation.parquet")

# **1.Feature Engineering**

In [ ]:
# Check the number of unique customers, customer IDs, and orders to understand the relationships between the datasets
print('Customers unique ID:', customers['customer_unique_id'].nunique())
print('Customer IDs:', customers['customer_id'].nunique())
print('Orders:', orders['order_id'].nunique())
print('Customers in orders:', orders['customer_id'].nunique())

Customers unique ID: 96096
Customer IDs: 99441
Orders: 99441
Customers in orders: 99441


In [ ]:
# Check the distribution of orders across different statuses
customer_orders['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [ ]:
# Identify the start and end dates of the order data.
print(customer_orders['order_purchase_timestamp'].min())
print(customer_orders['order_purchase_timestamp'].max())

2016-09-04 21:15:19
2018-10-17 17:30:18


# *1.1.Order-Level Features*

**Each row represents one order, linked to its corresponding unique customer.**

In [ ]:
# Creating "Order" dataframe through merging tables
orders_customers = orders.merge(customers[['customer_id', 'customer_unique_id']],
    on='customer_id',how='left')

In [ ]:
# Validate the merge by checking the total number of rows, missing customer IDs, and unique customers
print('Rows:', len(orders_customers))
print('Missing customers:', orders_customers['customer_unique_id'].isna().sum())
print('Unique customers:', orders_customers['customer_unique_id'].nunique())

Rows: 99441
Missing customers: 0
Unique customers: 96096


In [ ]:
# Filter delivered orders for customer purchase behavior analysis
delivered_orders = orders_customers[customer_orders['order_status'] == 'delivered'].copy()
delivered_orders['order_status'].value_counts()

,count
order_status,
delivered,96478


# *1.2.Customer-Level Features*

**Each row represents one customer, summarized based on their delivered orders.**

In [ ]:
# Calculate the total number of delivered orders per customer.
customer_purchase_count = (
    delivered_orders.groupby('customer_unique_id')['order_id']
    .nunique()
    .reset_index(name='purchase_count'))
customer_purchase_count.head()

,customer_unique_id,purchase_count
0,0000366f3b9a7992bf8c76cfdf3221e2,1
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1
2,0000f46a3911fa3c0805444483337064,1
3,0000f6ccb0745a6a4b88665a16c9f078,1
4,0004aac84e0df4da2b147fca70cf8255,1


In [ ]:
# Analyze the distribution of purchase frequency across customers.
customer_purchase_count['purchase_count'].value_counts().sort_index()

,count
purchase_count,
1,90557
2,2573
3,181
4,28
5,9
6,5
7,3
9,1
15,1


# ***Model 1: Repeat Purchase Prediction***
# Predict whether a first-time customer will make another purchase within the next 90 days.